# 01 - Data Inspection and Cleaning
Goal: understand the HR dataset, find quality issues, and decide how to handle them.

In [1]:
import pandas as pd, glob, os

for f in glob.glob("*.csv"):
    df = pd.read_csv(f)
    print("=" * 60)
    print(os.path.basename(f), df.shape)
    print(list(df.columns))
    print("Missing:", df.isna().sum()[df.isna().sum() > 0].to_dict())
    print("Duplicates:", df.duplicated().sum())
    print(df.head(2).to_string())

employee_data.csv (3000, 26)
['EmpID', 'FirstName', 'LastName', 'StartDate', 'ExitDate', 'Title', 'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeStatus', 'EmployeeType', 'PayZone', 'EmployeeClassificationType', 'TerminationType', 'TerminationDescription', 'DepartmentType', 'Division', 'DOB', 'State', 'JobFunctionDescription', 'GenderCode', 'LocationCode', 'RaceDesc', 'MaritalDesc', 'Performance Score', 'Current Employee Rating']
Missing: {'ExitDate': 1467, 'TerminationDescription': 1467}
Duplicates: 0
   EmpID FirstName LastName  StartDate ExitDate                    Title       Supervisor                      ADEmail BusinessUnit EmployeeStatus EmployeeType PayZone EmployeeClassificationType TerminationType TerminationDescription     DepartmentType              Division         DOB State JobFunctionDescription GenderCode  LocationCode  RaceDesc MaritalDesc Performance Score  Current Employee Rating
0   3427     Uriah  Bridges  20-Sep-19      NaN  Production Technician I     Peter O

In [2]:
import pandas as pd, glob, os

files = {os.path.basename(f)[:-4]: pd.read_csv(f) for f in glob.glob("*.csv")}
print("Files loaded:", list(files))

for name, df in files.items():
    print("\n" + "=" * 50, "\n", name, df.shape)
    print("Date-like columns:", [c for c in df.columns if "date" in c.lower()])
    for c in df.columns[~df.columns.isin(df.select_dtypes("number").columns)]:
        n = df[c].nunique()
        if n <= 15:
            print(f"  {c} ({n} values):", df[c].value_counts().head(8).to_dict())

Files loaded: ['employee_data', 'employee_engagement_survey_data', 'recruitment_data', 'training_and_development_data']

 employee_data (3000, 26)
Date-like columns: ['StartDate', 'ExitDate']
  BusinessUnit (10 values): {'NEL': 304, 'SVG': 304, 'BPC': 303, 'EW': 302, 'PL': 301, 'CCDR': 300, 'PYZ': 299, 'TNS': 297}
  EmployeeStatus (5 values): {'Active': 2458, 'Voluntarily Terminated': 321, 'Leave of Absence': 86, 'Future Start': 69, 'Terminated for Cause': 66}
  EmployeeType (3 values): {'Full-Time': 1038, 'Contract': 1008, 'Part-Time': 954}
  PayZone (3 values): {'Zone A': 1062, 'Zone B': 985, 'Zone C': 953}
  EmployeeClassificationType (3 values): {'Temporary': 1033, 'Full-Time': 1013, 'Part-Time': 954}
  TerminationType (5 values): {'Unk': 1467, 'Involuntary': 388, 'Voluntary': 388, 'Resignation': 380, 'Retirement': 377}
  DepartmentType (6 values): {'Production       ': 2020, 'IT/IS': 430, 'Sales': 331, 'Software Engineering': 115, 'Admin Offices': 80, 'Executive Office': 24}
  Gen

In [3]:
emp = files["employee_data"].copy()
print(list(emp.columns), "\n")
print(emp[["StartDate", "ExitDate"]].head(3), "\n")

for c in ["StartDate", "ExitDate"]:
    before = emp[c].isna().sum()
    emp[c] = pd.to_datetime(emp[c], errors="coerce")
    print(c, "missing before:", before, "| after:", emp[c].isna().sum())

print("\nExit before start:", (emp.ExitDate < emp.StartDate).sum())
print(pd.crosstab(emp["EmployeeStatus"], emp["TerminationType"]))

['EmpID', 'FirstName', 'LastName', 'StartDate', 'ExitDate', 'Title', 'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeStatus', 'EmployeeType', 'PayZone', 'EmployeeClassificationType', 'TerminationType', 'TerminationDescription', 'DepartmentType', 'Division', 'DOB', 'State', 'JobFunctionDescription', 'GenderCode', 'LocationCode', 'RaceDesc', 'MaritalDesc', 'Performance Score', 'Current Employee Rating'] 

   StartDate ExitDate
0  20-Sep-19      NaN
1  11-Feb-23      NaN
2  10-Dec-18      NaN 



C:\Users\bansa\AppData\Local\Temp\ipykernel_9456\495722066.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  emp[c] = pd.to_datetime(emp[c], errors="coerce")


StartDate missing before: 0 | after: 0
ExitDate missing before: 1467 | after: 1467

Exit before start: 0
TerminationType         Involuntary  Resignation  Retirement   Unk  Voluntary
EmployeeStatus                                                               
Active                          244          252         244  1467        251
Future Start                     18           11          23     0         17
Leave of Absence                 19           21          24     0         22
Terminated for Cause             21           22          10     0         13
Voluntarily Terminated           86           74          76     0         85


C:\Users\bansa\AppData\Local\Temp\ipykernel_9456\495722066.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  emp[c] = pd.to_datetime(emp[c], errors="coerce")


In [5]:
import pandas as pd, os

emp = pd.read_csv("employee_data.csv")

# 1. Strip spaces from text columns
for c in emp.select_dtypes(exclude="number").columns:
    emp[c] = emp[c].str.strip()

# 2. Dates with exact format
emp["StartDate"] = pd.to_datetime(emp["StartDate"], format="%d-%b-%y")
emp["ExitDate"]  = pd.to_datetime(emp["ExitDate"],  format="%d-%b-%y", errors="coerce")

# 3. Snapshot date comes from the data, not from today
snapshot = max(emp.StartDate.max(), emp.ExitDate.max())

# 4. Rule: left = has an ExitDate
emp["left"] = emp.ExitDate.notna()
emp["status_conflict"] = emp.left & ~emp.EmployeeStatus.isin(["Voluntarily Terminated", "Terminated for Cause"])

# 5. Tenure up to exit date, or the snapshot for current employees
emp["tenure_yrs"] = ((emp.ExitDate.fillna(snapshot) - emp.StartDate).dt.days / 365.25).round(1)
emp["tenure_band"] = pd.cut(emp.tenure_yrs, [-0.01, 1, 3, 5, 100], labels=["0-1", "1-3", "3-5", "5+"])

# 6. Privacy
emp = emp.drop(columns=["FirstName", "LastName", "ADEmail"])

# 7. Checks
print("Snapshot date:", snapshot.date())
print("Exit date and TerminationType agree:", (emp.left == (emp.TerminationType != "Unk")).all())
print("Current headcount:", (~emp.left).sum(), "| Left:", emp.left.sum(), "| Conflicts with status:", emp.status_conflict.sum())
print("Negative tenure:", (emp.tenure_yrs < 0).sum(), "\n")

rows = []
for y in range(2018, 2024):
    ye = pd.Timestamp(f"{y}-12-31") if y < 2023 else snapshot
    rows.append({"year": y,
                 "hires": (emp.StartDate.dt.year == y).sum(),
                 "exits": (emp.ExitDate.dt.year == y).sum(),
                 "headcount_end": ((emp.StartDate <= ye) & (emp.ExitDate.isna() | (emp.ExitDate > ye))).sum()})
print(pd.DataFrame(rows), "\n")
print(emp.groupby("DepartmentType").left.agg(["count", "mean"]).round(3), "\n")
print("DOB sample:", emp.DOB.head(3).tolist())

# 8. Save
os.makedirs("clean", exist_ok=True)
emp.to_csv("clean/employee_clean.csv", index=False)

Snapshot date: 2023-08-06
Exit date and TerminationType agree: True
Current headcount: 1467 | Left: 1533 | Conflicts with status: 1146
Negative tenure: 0 

   year  hires  exits  headcount_end
0  2018    255      4            251
1  2019    598     62            787
2  2020    592    133           1246
3  2021    600    278           1568
4  2022    620    460           1728
5  2023    335    596           1467 

                      count   mean
DepartmentType                    
Admin Offices            80  0.600
Executive Office         24  0.792
IT/IS                   430  0.521
Production             2020  0.502
Sales                   331  0.495
Software Engineering    115  0.557 

DOB sample: ['07-10-1969', '30-08-1965', '06-10-1991']


In [6]:
import pandas as pd

emp = pd.read_csv("clean/employee_clean.csv", parse_dates=["StartDate", "ExitDate"])
snapshot = pd.Timestamp("2023-08-06")

emp["DOB"] = pd.to_datetime(emp["DOB"], format="%d-%m-%Y")
emp["age"] = (emp.ExitDate.fillna(snapshot) - emp.DOB).dt.days // 365.25
order = ["<30", "30-39", "40-49", "50-59", "60+"]
emp["age_band"] = pd.cut(emp.age, [0, 29, 39, 49, 59, 100], labels=order).astype(str)
emp = emp.drop(columns=["DOB"])
emp.to_csv("clean/employee_final.csv", index=False)
print(emp.age.describe().round(1))
print(emp.age_band.value_counts().reindex(order))

count    3000.0
mean       50.8
std        17.7
min        18.0
25%        35.0
50%        50.0
75%        66.0
max        81.0
Name: age, dtype: float64
age_band
<30       470
30-39     495
40-49     482
50-59     452
60+      1101
Name: count, dtype: int64
